<a href="https://colab.research.google.com/github/alortiz05/DSBC_Cohort16-Projects/blob/main/SQL_2_BQ_project_templateALO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project SQL

## Pick a dataset that interests you (or multiple data sets)

Use the Open Data Sets available from Google BigQuery. You can use your own Google account or Kaggle.



In [ ]:
from google.cloud import bigquery
from google.colab import auth
import pandas as pd
from google.colab import userdata

auth.authenticate_user()

In [ ]:
#bbc_project_id = 'shaped-repeater-455122-d6'

In [ ]:
project3_id=userdata.get('bq_billing_project_id')
project3_id

'shaped-repeater-455122-d6'

In [ ]:
client = bigquery.Client(project=project3_id)

## Come up with questions about your data
* What sort of information is in this dataset?
* How many records are there?
* Have the number of bitcoin transactions increased year over year?
* Does New Mexico get more or less rain now than 20 years ago?
* How many different countries (states, counties, cities, etc) have records in this data set?




## Use SQL queries to pull specific information

Do NOT pull all the data and then filter using DataFrame methods etc. Make sure and use AT LEAST 13 of the 15 SQL options listed below. (You may have to get creative and come up with more questions to ask/answer.)

### Basic Queries


In [ ]:
# Construct a reference to the "bbc news" dataset
dataset_ref = client.dataset("bbc_news", project="bigquery-public-data")

# API request - fetch the dataset
dataset = client.get_dataset(dataset_ref)

# Get all the tables in the dataset
tables = list(client.list_tables(dataset))

# Print names of all tables in the dataset
for table in tables:
  print(table.table_id)

fulltext


In [ ]:
table_id = 'fulltext'
table_id

'fulltext'

In [ ]:
 #Construct a reference to the "mobility report" table
table_ref = dataset.table("fulltext")

# API request - fetch the table
table = client.get_table(table_ref)

# See the table's schema - name, field type, mode, description
table.schema

[SchemaField('body', 'STRING', 'NULLABLE', None, None, (), None),
 SchemaField('title', 'STRING', 'NULLABLE', None, None, (), None),
 SchemaField('filename', 'STRING', 'NULLABLE', None, None, (), None),
 SchemaField('category', 'STRING', 'NULLABLE', None, None, (), None)]

In [ ]:
# convert the table.schema into a data frame
fields = pd.DataFrame( [ x.to_api_repr() for x in table.schema ] )
fields.head()


,name,type,mode
0,body,STRING,NULLABLE
1,title,STRING,NULLABLE
2,filename,STRING,NULLABLE
3,category,STRING,NULLABLE


In [ ]:
fields.shape

(4, 3)

In [ ]:
# Preview the first five lines of the table as a data frame
df=client.list_rows(table, max_results=10).to_dataframe()
df.head()

,body,title,filename,category
0,The global web blog community is being called ...,Global blogger action day called,bbc/tech/016.txt,tech
1,"The ""digital divide"" between rich and poor nat...",Global digital divide 'narrowing',bbc/tech/033.txt,tech
2,The current slew of sports games offers unpara...,Sporting rivals go to extra time,bbc/tech/056.txt,tech
3,Writing a Microsoft Word document can be a dan...,Warning over Windows Word files,bbc/tech/086.txt,tech
4,"Aid workers trying to house, feed and clothe m...",Satellite mapping aids Darfur relief,bbc/tech/223.txt,tech


In [ ]:
client = bigquery.Client(project=project_id)

row_count = client.query('''
  SELECT
    COUNT(1) as total
  FROM `bigquery-public-data.bbc_news.fulltext`
  '''
).to_dataframe()["total"][0]

print(f'Full dataset has {row_count:_} rows')

Full dataset has 2_225 rows


Set constants for sizes

In [ ]:
ONE_MB = 1_000*1_000
ONE_GB = 1_000*ONE_MB
TWO_GB = 2*ONE_GB

In [ ]:
project_id = "bigquery-public-data"
dataset_id = "bbc_news"
table_id = "fulltext"

queries = []

queries += [ f"""
        SELECT category
        FROM {project_id}.{dataset_id}.{table_id}
        LIMIT 10
        """ ]

queries += [ f"""
        SELECT category, COUNT(*) AS value_count
        FROM {project_id}.{dataset_id}.{table_id}
        GROUP BY category
        ORDER BY value_count DESC
        """ ]

queries += [ f"""
        SELECT DISTINCT category,
        FROM {project_id}.{dataset_id}.{table_id}
        """ ]

queries += [ f"""
        SELECT category
        FROM {project_id}.{dataset_id}.{table_id}
        WHERE category = 'business'
        """ ]

queries += [ f"""
        WITH words AS (
          SELECT
            LOWER(word) AS word
          FROM {project_id}.{dataset_id}.{table_id}
          CROSS JOIN UNNEST(REGEXP_EXTRACT_ALL(title, r'\w+')) AS word
          WHERE
            category = 'business'
        )
        SELECT
          word,
          COUNT(*) AS word_count
        FROM
          words
        GROUP BY word
        ORDER BY word_count DESC
        LIMIT 10
        """ ]


len(queries)

5

#### SELECT (with * and with column names)


In [ ]:
for query in queries:
  dry_run_config = bigquery.QueryJobConfig(dry_run = True)
  dry_run_query_job = client.query(query, job_config= dry_run_config)
  size = dry_run_query_job.total_bytes_processed
  print(query)
  print(f"{size:_}")
  print()

  #limit does not limit the data you touch but only how much you will see


        SELECT category
        FROM bigquery-public-data.bbc_news.fulltext
        LIMIT 10
        
21_043


        SELECT category, COUNT(*) AS value_count
        FROM bigquery-public-data.bbc_news.fulltext
        GROUP BY category
        ORDER BY value_count DESC
        
21_043


        SELECT DISTINCT category,
        FROM bigquery-public-data.bbc_news.fulltext
        
21_043


        SELECT category
        FROM bigquery-public-data.bbc_news.fulltext
        WHERE category = 'business'
        
21_043


        WITH words AS (
          SELECT
            LOWER(word) AS word
          FROM bigquery-public-data.bbc_news.fulltext
          CROSS JOIN UNNEST(REGEXP_EXTRACT_ALL(title, r'\w+')) AS word
          WHERE
            category = 'business'
        )
        SELECT
          word,
          COUNT(*) AS word_count
        FROM
          words
        GROUP BY word
        ORDER BY word_count DESC
        LIMIT 10
        
95_295



In [ ]:
# safe_config needs to be included with every client.query() request
safe_config = bigquery.QueryJobConfig(
    maximum_bytes_billed=TWO_GB
    # totalBytesProcessed=ONE_GB,
    # total_bytes_processed=ONE_GB,
)
# Use a try...except block to catch when the safe_config paramenter prevents a query
for query in queries:
  print(query)
  try:
    df = client.query(query, job_config=safe_config).to_dataframe()
    print(df.head())
  except:
    print("Blocked by safe_config")


        SELECT category
        FROM bigquery-public-data.bbc_news.fulltext
        LIMIT 10
        
  category
0     tech
1     tech
2     tech
3     tech
4     tech

        SELECT category, COUNT(*) AS value_count
        FROM bigquery-public-data.bbc_news.fulltext
        GROUP BY category
        ORDER BY value_count DESC
        
        category  value_count
0          sport          511
1       business          510
2       politics          417
3           tech          401
4  entertainment          386

        SELECT DISTINCT category,
        FROM bigquery-public-data.bbc_news.fulltext
        
        category
0           tech
1          sport
2       business
3       politics
4  entertainment

        SELECT category
        FROM bigquery-public-data.bbc_news.fulltext
        WHERE category = 'business'
        
   category
0  business
1  business
2  business
3  business
4  business

        WITH words AS (
          SELECT
            LOWER(word) AS word
          FROM

#### WHERE


used

#### AND


#### OR


#### LIKE (with % or _ wildcard)


#### BETWEEN


#### LIMIT



### Sorting and Grouping


#### ORDER BY


used

#### DISTINCT


#### GROUP BY



used

### Aggregates


#### MAX


#### MIN


#### SUM


#### AVG


#### COUNT



## Make some plots

Make some cool plots to go with your data. Write SQL queries to get ONLY the information you need for each plot. (Don't pull ALL the data and then just plot a few columns.)



## EXTRA CREDIT:

#### Use a query that joins two tables.


#### Make a model to see if you can predict something


#### Come up with something else cool to do with your data
